## Data Cleaning Pipeline

### 1. Load the necessary packages

In [128]:
import pandas as pd
import numpy as np
from pathlib import Path

#### Set up the path for working directory

In [199]:
# Input folder containing the combined residential datasets
INPUT_DIR = Path("output")

# Output folder for cleaned datasets
CLEANED_DIR = Path("cleaned_data")
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

print("Input folder:", INPUT_DIR)
print("Cleaned output folder:", CLEANED_DIR)

Input folder: output
Cleaned output folder: cleaned_data


### 2. Read the datasets

In [130]:
# (combined_sold/list_residential_with_rate.csv)

sold_raw = pd.read_csv(
    INPUT_DIR / "combined_sold_residential_with_mortgage_rates.csv",
    low_memory=False
)

list_raw = pd.read_csv(
    INPUT_DIR / "combined_list_residential_with_mortgage_rates.csv",
    low_memory=False
)

print("Sold raw shape:", sold_raw.shape)
print("List raw shape:", list_raw.shape)

Sold raw shape: (448033, 89)
List raw shape: (615739, 89)


#### Copy the original dataset

In [131]:
sold_clean = sold_raw.copy()
list_clean = list_raw.copy()

print("Sold clean shape:", sold_clean.shape)
print("List clean shape:", list_clean.shape)

Sold clean shape: (448033, 89)
List clean shape: (615739, 89)


### 3. Create the cleaning log to document any change in dataset

In [132]:
cleaning_log = []


def add_cleaning_log(
    dataset_name,
    step,
    rows_before,
    rows_after,
    cols_before,
    cols_after,
    notes=""
):
    cleaning_log.append({
        "dataset": dataset_name,
        "step": step,
        "rows_before": rows_before,
        "rows_after": rows_after,
        "rows_removed": rows_before - rows_after,
        "columns_before": cols_before,
        "columns_after": cols_after,
        "columns_removed": cols_before - cols_after,
        "notes": notes
    })

### 4. Initial Data Inspection

In [133]:
# Check data shape, columns, and data type
def inspect_dataset(df, dataset_name):
    print("\n" + "-" * 60)
    print(dataset_name)
    print("-" * 60)

    print("Shape:", df.shape)
    print("\nData types:")
    print(df.dtypes.value_counts())

    print("\nDuplicate full rows:")
    print(df.duplicated().sum())

    print("\nTotal missing values:")
    print(df.isna().sum().sum())


inspect_dataset(sold_clean, "Sold Dataset")
inspect_dataset(list_clean, "List Dataset")


------------------------------------------------------------
Sold Dataset
------------------------------------------------------------
Shape: (448033, 89)

Data types:
object     56
float64    30
int64       3
Name: count, dtype: int64

Duplicate full rows:
48

Total missing values:
12864595

------------------------------------------------------------
List Dataset
------------------------------------------------------------
Shape: (615739, 89)

Data types:
object     51
float64    34
int64       4
Name: count, dtype: int64

Duplicate full rows:
0

Total missing values:
18752296


In [134]:
# Check dataset time range is from 202401 to 202606
print("Sold year_month range:")
print(sold_clean["year_month"].min(), sold_clean["year_month"].max())

print("\nList year_month range:")
print(list_clean["year_month"].min(), list_clean["year_month"].max())

Sold year_month range:
2024-01 2026-06

List year_month range:
2024-01 2026-06


In [135]:
# Check each month csv file rows count in sold dataset
print(
    sold_clean["year_month"]
    .value_counts(dropna=False)
    .sort_index()
    .to_string()
)

year_month
2024-01    11203
2024-02    13063
2024-03    15784
2024-04    17295
2024-05    18474
2024-06    16625
2024-07    17809
2024-08    16791
2024-09    14555
2024-10    16275
2024-11    14158
2024-12    14220
2025-01    10976
2025-02    12101
2025-03    14409
2025-04    15851
2025-05    15544
2025-06    15329
2025-07    15800
2025-08    15225
2025-09    15134
2025-10    15897
2025-11    12978
2025-12    13975
2026-01    10083
2026-02    12272
2026-03    15656
2026-04    16581
2026-05    16384
2026-06    17586


In [136]:
# Check each month csv file rows count in list dataset
print(
    list_clean["year_month"]
    .value_counts(dropna=False)
    .sort_index()
    .to_string()
)

year_month
2024-01    17007
2024-02    17490
2024-03    20501
2024-04    24025
2024-05    25447
2024-06    23310
2024-07    23019
2024-08    22215
2024-09    22257
2024-10    21921
2024-11    15108
2024-12    10694
2025-01    22690
2025-02    21930
2025-03    25104
2025-04    26695
2025-05    26438
2025-06    17056
2025-07    17194
2025-08    15658
2025-09    16859
2025-10    17186
2025-11    12194
2025-12    10516
2026-01    22108
2026-02    20823
2026-03    25614
2026-04    26490
2026-05    24019
2026-06    24171


### 5. Standardize Column Names

In [201]:
sold_clean.columns = sold_clean.columns.str.strip()
list_clean.columns = list_clean.columns.str.strip()

print("Sold Clean Columns List")
print(sold_clean.columns.tolist())

print("List Clean Columns List")
print(list_clean.columns.tolist())

Sold Clean Columns List
['BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'PoolPrivateYN', 'OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate', 'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude', 'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea', 'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'AssociationFeeFrequency', 'ListingKeyNumeric', 'MLSAreaMajor', 'CountyOrParish', 'MlsStatus', 'ElementarySchool', 'AttachedGarageYN', 'ParkingTotal', 'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt', 'StreetNumberNumeric', 'ListingId', 'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'ContractStatusChangeDate', 'PurchaseContractDate', 'ListingContractDate', 'StateOrProvince', 'MiddleOrJuniorSchool', 'FireplaceYN', 'Stories', 'HighSchool', 'Levels'

### 6. Standardize Data Types

#### 6.1 Convert Date Columns

In [138]:
date_columns = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate"
]

In [139]:
def convert_date_columns(df, date_columns, dataset_name):
    df = df.copy()

    for col in date_columns:
        if col in df.columns:
            original_non_null = df[col].notna().sum()

            df[col] = pd.to_datetime(
                df[col],
                errors="coerce"
            )

            converted_non_null = df[col].notna().sum()
            invalid_count = original_non_null - converted_non_null

            print(
                f"{dataset_name} | {col}: "
                f"{invalid_count:,} values became NaT"
            )
        else:
            print(f"{dataset_name} | {col}: column not found")

    return df

In [140]:
sold_clean = convert_date_columns(
    sold_clean,
    date_columns,
    "Sold"
)

list_clean = convert_date_columns(
    list_clean,
    date_columns,
    "List"
)

Sold | CloseDate: 0 values became NaT
Sold | PurchaseContractDate: 0 values became NaT
Sold | ListingContractDate: 0 values became NaT
Sold | ContractStatusChangeDate: 0 values became NaT
List | CloseDate: 0 values became NaT
List | PurchaseContractDate: 0 values became NaT
List | ListingContractDate: 0 values became NaT
List | ContractStatusChangeDate: 0 values became NaT


In [141]:
# check the date cols' data type is datetime
for col in date_columns:
    if col in sold_clean.columns:
        print(col, sold_clean[col].dtype)

CloseDate datetime64[ns]
PurchaseContractDate datetime64[ns]
ListingContractDate datetime64[ns]
ContractStatusChangeDate datetime64[ns]


#### 6.2 Convert boolean columns

In [142]:
boolean_columns = [
    "AttachedGarageYN",
    "ViewYN",
    "PoolPrivateYN",
    "NewConstructionYN",
    "FireplaceYN",
    "WaterfrontYN",
    "BasementYN"
]

In [143]:
print("Sold boolean column dtypes:")
print(
    sold_clean[
        [col for col in boolean_columns if col in sold_clean.columns]
    ].dtypes
)

print("\nList boolean column dtypes:")
print(
    list_clean[
        [col for col in boolean_columns if col in list_clean.columns]
    ].dtypes
)

Sold boolean column dtypes:
AttachedGarageYN     object
ViewYN               object
PoolPrivateYN        object
NewConstructionYN    object
FireplaceYN          object
WaterfrontYN         object
BasementYN           object
dtype: object

List boolean column dtypes:
AttachedGarageYN     object
NewConstructionYN    object
FireplaceYN          object
dtype: object


In [202]:
# To verify they are boolean
for col in boolean_columns:
    if col in sold_clean.columns:
        print("\n" + "-" * 60)
        print(f"Sold | {col}")
        print(sold_clean[col].map(type).value_counts())

for col in boolean_columns:
    if col in list_clean.columns:
        print("\n" + "-" * 60)
        print(f"List | {col}")
        print(list_clean[col].map(type).value_counts())

# It should have these results
# <class 'bool'>
# <class 'pandas._libs.missing.NAType'>


------------------------------------------------------------
Sold | AttachedGarageYN
AttachedGarageYN
<class 'bool'>                           379411
<class 'pandas._libs.missing.NAType'>     68358
Name: count, dtype: int64

------------------------------------------------------------
Sold | ViewYN
ViewYN
<class 'bool'>                           409357
<class 'pandas._libs.missing.NAType'>     38412
Name: count, dtype: int64

------------------------------------------------------------
Sold | PoolPrivateYN
PoolPrivateYN
<class 'bool'>                           409549
<class 'pandas._libs.missing.NAType'>     38220
Name: count, dtype: int64

------------------------------------------------------------
Sold | NewConstructionYN
NewConstructionYN
<class 'bool'>                           414373
<class 'pandas._libs.missing.NAType'>     33396
Name: count, dtype: int64

------------------------------------------------------------
Sold | FireplaceYN
FireplaceYN
<class 'bool'>                 

In [145]:
# Convert them into boolean
for col in boolean_columns:
    if col in sold_clean.columns:
        sold_clean[col] = sold_clean[col].astype("boolean")

for col in boolean_columns:
    if col in list_clean.columns:
        list_clean[col] = list_clean[col].astype("boolean")

In [146]:
# Verify conversion (The data type shouldn't be object)
print("Sold boolean column dtypes after conversion:")
print(
    sold_clean[
        [col for col in boolean_columns if col in sold_clean.columns]
    ].dtypes
)

print("\nList boolean column dtypes after conversion:")
print(
    list_clean[
        [col for col in boolean_columns if col in list_clean.columns]
    ].dtypes
)

Sold boolean column dtypes after conversion:
AttachedGarageYN     boolean
ViewYN               boolean
PoolPrivateYN        boolean
NewConstructionYN    boolean
FireplaceYN          boolean
WaterfrontYN         boolean
BasementYN           boolean
dtype: object

List boolean column dtypes after conversion:
AttachedGarageYN     boolean
NewConstructionYN    boolean
FireplaceYN          boolean
dtype: object


#### 6.3 Convert Numeric Columns

In [147]:
# define the numeric columns
numeric_columns = [
    "ClosePrice",
    "ListPrice",
    "OriginalListPrice",
    "LivingArea",
    "LotSizeAcres",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "BathroomsFull",
    "BathroomsHalf",
    "DaysOnMarket",
    "YearBuilt",
    "Latitude",
    "Longitude",
    "TaxAnnualAmount",
    "ParkingTotal",
    "FireplacesTotal",
    "BuildingAreaTotal",
    "AboveGradeFinishedArea"
]

In [148]:
# create a function
def convert_numeric_columns(df, numeric_columns, dataset_name):
    df = df.copy()
    conversion_summary = []

    for col in numeric_columns:
        if col not in df.columns:
            continue

        original_non_null = df[col].notna().sum()
        original_dtype = str(df[col].dtype)

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        converted_non_null = df[col].notna().sum()

        conversion_summary.append({
            "dataset": dataset_name,
            "column": col,
            "original_dtype": original_dtype,
            "new_dtype": str(df[col].dtype),
            "values_converted_to_nan":
                original_non_null - converted_non_null
        })

    return df, pd.DataFrame(conversion_summary)

In [149]:
sold_clean, sold_numeric_conversion = convert_numeric_columns(
    sold_clean,
    numeric_columns,
    "Sold"
)

list_clean, list_numeric_conversion = convert_numeric_columns(
    list_clean,
    numeric_columns,
    "List"
)

In [150]:
print(sold_numeric_conversion.to_string(index=False))
print()
print(list_numeric_conversion.to_string(index=False))

dataset                 column original_dtype new_dtype  values_converted_to_nan
   Sold             ClosePrice        float64   float64                        0
   Sold              ListPrice        float64   float64                        0
   Sold      OriginalListPrice        float64   float64                        0
   Sold             LivingArea        float64   float64                        0
   Sold           LotSizeAcres        float64   float64                        0
   Sold          BedroomsTotal        float64   float64                        0
   Sold  BathroomsTotalInteger        float64   float64                        0
   Sold           DaysOnMarket          int64     int64                        0
   Sold              YearBuilt        float64   float64                        0
   Sold               Latitude        float64   float64                        0
   Sold              Longitude        float64   float64                        0
   Sold        TaxAnnualAmou

### 7. Remove Duplicate Columns

#### Find out the duplicate columns

In [ ]:
sold_dot_one_columns = [
    col for col in sold_clean.columns
    if col.endswith(".1")
]

list_dot_one_columns = [
    col for col in list_clean.columns
    if col.endswith(".1")
]

print("Sold .1 columns:")
print(sold_dot_one_columns)

print("\nList .1 columns:")
print(list_dot_one_columns)

Sold .1 columns:
[]

List .1 columns:
['PropertyType.1', 'ListAgentFirstName.1', 'DaysOnMarket.1', 'LivingArea.1', 'Longitude.1', 'Latitude.1', 'ListPrice.1', 'ListAgentLastName.1', 'CloseDate.1', 'BuyerOfficeName.1', 'UnparsedAddress.1']


#### Compare original columns and .1 columns

In [152]:
def compare_dot_one_columns(df):
    comparison_results = []

    dot_one_columns = [
        col for col in df.columns
        if col.endswith(".1")
    ]

    for duplicate_col in dot_one_columns:
        original_col = duplicate_col[:-2]

        if original_col not in df.columns:
            comparison_results.append({
                "original_column": original_col,
                "duplicate_column": duplicate_col,
                "original_exists": False,
                "same_values": False,
                "different_rows": None
            })
            continue

        same_mask = (
            df[original_col].eq(df[duplicate_col]) |
            (
                df[original_col].isna() &
                df[duplicate_col].isna()
            )
        )

        comparison_results.append({
            "original_column": original_col,
            "duplicate_column": duplicate_col,
            "original_exists": True,
            "same_values": bool(same_mask.all()),
            "different_rows": int((~same_mask).sum())
        })

    return pd.DataFrame(comparison_results)

In [153]:
sold_duplicate_column_check = compare_dot_one_columns(sold_clean)
list_duplicate_column_check = compare_dot_one_columns(list_clean)

print("Sold duplicate-column comparison:")
print(sold_duplicate_column_check.to_string(index=False))

print("\nList duplicate-column comparison:")
print(list_duplicate_column_check.to_string(index=False))

Sold duplicate-column comparison:
Empty DataFrame
Columns: []
Index: []

List duplicate-column comparison:
   original_column     duplicate_column  original_exists  same_values  different_rows
      PropertyType       PropertyType.1             True         True               0
ListAgentFirstName ListAgentFirstName.1             True         True               0
      DaysOnMarket       DaysOnMarket.1             True         True               0
        LivingArea         LivingArea.1             True         True               0
         Longitude          Longitude.1             True         True               0
          Latitude           Latitude.1             True         True               0
         ListPrice          ListPrice.1             True         True               0
 ListAgentLastName  ListAgentLastName.1             True         True               0
         CloseDate          CloseDate.1             True         True               0
   BuyerOfficeName    BuyerOffice

#### Only delete the duplicate columns have the same values

In [154]:
list_duplicate_columns_to_drop = (
    list_duplicate_column_check
    .loc[
        list_duplicate_column_check["same_values"],
        "duplicate_column"
    ]
    .tolist()
)

print("List columns to drop:")
print(list_duplicate_columns_to_drop)

List columns to drop:
['PropertyType.1', 'ListAgentFirstName.1', 'DaysOnMarket.1', 'LivingArea.1', 'Longitude.1', 'Latitude.1', 'ListPrice.1', 'ListAgentLastName.1', 'CloseDate.1', 'BuyerOfficeName.1', 'UnparsedAddress.1']


#### Delete duplicate columns and document it into cleaning_log

In [155]:
rows_before, cols_before = list_clean.shape

list_clean = list_clean.drop(
    columns=list_duplicate_columns_to_drop
)

rows_after, cols_after = list_clean.shape

add_cleaning_log(
    dataset_name="List",
    step="Remove duplicate .1 columns",
    rows_before=rows_before,
    rows_after=rows_after,
    cols_before=cols_before,
    cols_after=cols_after,
    notes=f"Dropped: {list_duplicate_columns_to_drop}"
)

### 8. Missing Value Analysis

#### Missing Value Summary

In [156]:
# create missing value summary function
def create_missing_summary(df):
    summary = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "non_null_count": df.notna().sum().values,
        "missing_count": df.isna().sum().values,
        "missing_percent": (
            df.isna().mean() * 100
        ).round(2).values,
        "unique_count": df.nunique(dropna=True).values
    })

    return summary.sort_values(
        by="missing_percent",
        ascending=False
    ).reset_index(drop=True)

In [157]:
sold_missing_summary = create_missing_summary(sold_clean)
list_missing_summary = create_missing_summary(list_clean)

print("Sold missing summary:")
print(sold_missing_summary.to_string(index=False))

print("\nList missing summary:")
print(list_missing_summary.to_string(index=False))

Sold missing summary:
                      column          dtype  non_null_count  missing_count  missing_percent  unique_count
             TaxAnnualAmount        float64               0         448033           100.00             0
MiddleOrJuniorSchoolDistrict        float64               0         448033           100.00             0
                BusinessType        float64               0         448033           100.00             0
    ElementarySchoolDistrict        float64               0         448033           100.00             0
                     TaxYear        float64               0         448033           100.00             0
             FireplacesTotal        float64               0         448033           100.00             0
      AboveGradeFinishedArea        float64               0         448033           100.00             0
               CoveredSpaces        float64               0         448033           100.00             0
                Waterfro

#### Review columns with over 90% missing rate

In [158]:
sold_over_90_missing = sold_missing_summary[
    sold_missing_summary["missing_percent"] > 90
]

list_over_90_missing = list_missing_summary[
    list_missing_summary["missing_percent"] > 90
]

print("Sold columns over 90% missing:")
print(sold_over_90_missing.to_string(index=False))

print("\nList columns over 90% missing:")
print(list_over_90_missing.to_string(index=False))

Sold columns over 90% missing:
                      column   dtype  non_null_count  missing_count  missing_percent  unique_count
             TaxAnnualAmount float64               0         448033           100.00             0
MiddleOrJuniorSchoolDistrict float64               0         448033           100.00             0
                BusinessType float64               0         448033           100.00             0
    ElementarySchoolDistrict float64               0         448033           100.00             0
                     TaxYear float64               0         448033           100.00             0
             FireplacesTotal float64               0         448033           100.00             0
      AboveGradeFinishedArea float64               0         448033           100.00             0
               CoveredSpaces float64               0         448033           100.00             0
                WaterfrontYN boolean             287         447746           

#### Review columns with 50% to 90% missing rate

In [159]:
sold_50_to_90_missing = sold_missing_summary[
    sold_missing_summary["missing_percent"].between(
        50,
        90,
        inclusive="both"
    )
]

list_50_to_90_missing = list_missing_summary[
    list_missing_summary["missing_percent"].between(
        50,
        90,
        inclusive="both"
    )
]

print("Sold columns 50% to 90% missing:")
print(sold_50_to_90_missing.to_string(index=False))

print("\nList columns 50% to 90% missing:")
print(list_50_to_90_missing.to_string(index=False))

Sold columns 50% to 90% missing:
                     column   dtype  non_null_count  missing_count  missing_percent  unique_count
    BuyerAgencyCompensation float64           46125         401908            89.70           174
BuyerAgencyCompensationType  object           46136         401897            89.70             3
           ElementarySchool  object           59523         388510            86.71          1409
       MiddleOrJuniorSchool  object           59945         388088            86.62           662
                  lonfilled  object           63884         384149            85.74             2
                  latfilled  object           63884         384149            85.74             2
                 HighSchool  object           77857         370176            82.62           521
      OriginatingSystemName  object           89682         358351            79.98             1
   OriginatingSystemSubName  object           89682         358351            79.98  

#### Drop over 90% missing rate columns

In [160]:
# make over 90% missing rate columns into a list
sold_cols_to_drop = sold_missing_summary.loc[
    sold_missing_summary["missing_percent"] > 90,
    "column"
].tolist()

list_cols_to_drop = list_missing_summary.loc[
    list_missing_summary["missing_percent"] > 90,
    "column"
].tolist()

print("Sold columns to drop:")
print(sold_cols_to_drop)

print("\nList columns to drop:")
print(list_cols_to_drop)

Sold columns to drop:
['TaxAnnualAmount', 'MiddleOrJuniorSchoolDistrict', 'BusinessType', 'ElementarySchoolDistrict', 'TaxYear', 'FireplacesTotal', 'AboveGradeFinishedArea', 'CoveredSpaces', 'WaterfrontYN', 'BelowGradeFinishedArea', 'BasementYN', 'LotSizeDimensions', 'BuilderName', 'BuildingAreaTotal', 'CoBuyerAgentFirstName']

List columns to drop:
['TaxYear', 'BusinessType', 'ElementarySchoolDistrict', 'MiddleOrJuniorSchoolDistrict', 'TaxAnnualAmount', 'AboveGradeFinishedArea', 'FireplacesTotal', 'CoveredSpaces', 'BelowGradeFinishedArea', 'CoBuyerAgentFirstName', 'BuilderName', 'LotSizeDimensions', 'BuildingAreaTotal']


In [161]:
# drop over 90% missig rate columns
sold_clean = sold_clean.drop(
    columns=sold_cols_to_drop
)

list_clean = list_clean.drop(
    columns=list_cols_to_drop
)

In [162]:
# Verify the result
print("Sold shape after dropping columns:", sold_clean.shape)
print("List shape after dropping columns:", list_clean.shape)

Sold shape after dropping columns: (448033, 74)
List shape after dropping columns: (615739, 65)


In [163]:
# Verify if over 90% columns has been drop
print(
    "Sold dropped columns still present:",
    [col for col in sold_cols_to_drop if col in sold_clean.columns]
)

print(
    "List dropped columns still present:",
    [col for col in list_cols_to_drop if col in list_clean.columns]
)

Sold dropped columns still present: []
List dropped columns still present: []


### 9. Check Sold dataset 2 missing ClosePrice rows

In [164]:
sold_missing_close_price = sold_clean[
    sold_clean["ClosePrice"].isna()
].copy()

print("Sold rows with missing ClosePrice:")
print(sold_missing_close_price.shape)

display(sold_missing_close_price)

Sold rows with missing ClosePrice:
(2, 74)


,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,...,OriginatingSystemSubName,source_file,year_month,data_type,file_version,BuyerAgencyCompensationType,BuyerAgencyCompensation,latfilled,lonfilled,rate_30yr_fixed
92215,MariposaCounty,MariposaCounty,NaN,True,True,575000.0,1045745288,shelby.twissrealty@gmail.com,2024-06-06,NaN,...,NaN,CRMLSSold202406_filled.csv,2024-06,sold,filled,NaN,NaN,True,True,6.9175
182391,MercedCounty,MercedCounty,NaN,True,True,2500.0,1086646611,trustedrealtors4ucarla@gmail.com,2024-12-13,NaN,...,NaN,CRMLSSold202412.csv,2024-12,sold,original,NaN,NaN,NaN,NaN,6.7150


In [165]:
# Check the important point
columns_to_review = [
    "ListingKey",
    "ListingId",
    "ClosePrice",
    "ListPrice",
    "CloseDate",
    "PropertyType",
    "PropertySubType",
    "MlsStatus"
]

existing_columns = [
    col for col in columns_to_review
    if col in sold_missing_close_price.columns
]

display(
    sold_missing_close_price[existing_columns]
)

,ListingKey,ListingId,ClosePrice,ListPrice,CloseDate,PropertyType,PropertySubType,MlsStatus
92215,1045745288,MP23181752,NaN,575000.0,2024-06-06,Residential,SingleFamilyResidence,Closed
182391,1086646611,MC24191255,NaN,2700000.0,2024-12-13,Residential,SingleFamilyResidence,Closed


### 10. Geographic Data Checks

#### Set up California latitude and longitude boundary

In [166]:
CA_LAT_MIN = 32
CA_LAT_MAX = 42

CA_LON_MIN = -125
CA_LON_MAX = -114

In [167]:
# create a function for geographic flag
def add_geographic_flags(df):
    df = df.copy()

    df["lat_zero_flag"] = df["Latitude"].eq(0)

    df["long_zero_flag"] = df["Longitude"].eq(0)

    valid_coordinates = (
        df["Latitude"].notna() &
        df["Longitude"].notna()
    )

    df["out_of_state_flag"] = (
        valid_coordinates &
        (
            (df["Latitude"] < CA_LAT_MIN) |
            (df["Latitude"] > CA_LAT_MAX) |
            (df["Longitude"] < CA_LON_MIN) |
            (df["Longitude"] > CA_LON_MAX)
        )
    )

    return df

In [168]:
sold_clean = add_geographic_flags(sold_clean)
list_clean = add_geographic_flags(list_clean)

#### Check geographic flags

In [169]:
geographic_flags = [
    "lat_zero_flag",
    "long_zero_flag",
    "out_of_state_flag"
]

print("Sold geographic flags:")
print(
    sold_clean[geographic_flags]
    .sum()
    .to_string()
)

print("\nList geographic flags:")
print(
    list_clean[geographic_flags]
    .sum()
    .to_string()
)

Sold geographic flags:
lat_zero_flag        37
long_zero_flag       37
out_of_state_flag    99

List geographic flags:
lat_zero_flag         75
long_zero_flag        75
out_of_state_flag    323


### 10. Invalid Numeric Value Checks

#### Invalid numeric-values flag

In [170]:
def add_invalid_numeric_flags(df, dataset_type):
    df = df.copy()

    if dataset_type == "sold" and "ClosePrice" in df.columns:
        df["invalid_close_price_flag"] = (
            df["ClosePrice"].notna() &
            df["ClosePrice"].le(0)
        )

    if "ListPrice" in df.columns:
        df["invalid_list_price_flag"] = (
            df["ListPrice"].notna() &
            df["ListPrice"].le(0)
        )

    if "LivingArea" in df.columns:
        df["invalid_living_area_flag"] = (
            df["LivingArea"].notna() &
            df["LivingArea"].le(0)
        )

    if "DaysOnMarket" in df.columns:
        df["negative_days_on_market_flag"] = (
            df["DaysOnMarket"].notna() &
            df["DaysOnMarket"].lt(0)
        )

    if "BedroomsTotal" in df.columns:
        df["negative_bedrooms_flag"] = (
            df["BedroomsTotal"].notna() &
            df["BedroomsTotal"].lt(0)
        )

    if "BathroomsTotalInteger" in df.columns:
        df["negative_bathrooms_flag"] = (
            df["BathroomsTotalInteger"].notna() &
            df["BathroomsTotalInteger"].lt(0)
        )

    return df

In [171]:
sold_clean = add_invalid_numeric_flags(
    sold_clean,
    dataset_type="sold"
)

list_clean = add_invalid_numeric_flags(
    list_clean,
    dataset_type="list"
)

#### Check invalid-value counts

In [172]:
sold_invalid_flag_columns = [
    col for col in sold_clean.columns
    if col.startswith("invalid_") or col.startswith("negative_")
]

list_invalid_flag_columns = [
    col for col in list_clean.columns
    if col.startswith("invalid_") or col.startswith("negative_")
]

print("Sold invalid numeric flags:")
print(
    sold_clean[sold_invalid_flag_columns]
    .sum()
    .sort_values(ascending=False)
    .to_string()
)

print("\nList invalid numeric flags:")
print(
    list_clean[list_invalid_flag_columns]
    .sum()
    .sort_values(ascending=False)
    .to_string()
)

Sold invalid numeric flags:
invalid_living_area_flag        165
negative_days_on_market_flag     50
invalid_close_price_flag          1
invalid_list_price_flag           0
negative_bedrooms_flag            0
negative_bathrooms_flag           0

List invalid numeric flags:
invalid_living_area_flag        393
negative_days_on_market_flag     30
invalid_list_price_flag           0
negative_bedrooms_flag            0
negative_bathrooms_flag           0


### 11. Date consistency flags

In [173]:
# create a function
def add_date_consistency_flags(df):
    df = df.copy()

    required_columns = [
        "ListingContractDate",
        "PurchaseContractDate",
        "CloseDate"
    ]

    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        print(
            "Cannot create all date flags. Missing columns:",
            missing_columns
        )
        return df

    df["listing_after_close_flag"] = (
        df["ListingContractDate"].notna() &
        df["CloseDate"].notna() &
        (
            df["ListingContractDate"] >
            df["CloseDate"]
        )
    )

    df["purchase_after_close_flag"] = (
        df["PurchaseContractDate"].notna() &
        df["CloseDate"].notna() &
        (
            df["PurchaseContractDate"] >
            df["CloseDate"]
        )
    )

    df["negative_timeline_flag"] = (
        df["ListingContractDate"].notna() &
        df["PurchaseContractDate"].notna() &
        (
            df["PurchaseContractDate"] <
            df["ListingContractDate"]
        )
    )

    return df

In [174]:
sold_clean = add_date_consistency_flags(sold_clean)
list_clean = add_date_consistency_flags(list_clean)

In [175]:
date_flag_columns = [
    "listing_after_close_flag",
    "purchase_after_close_flag",
    "negative_timeline_flag"
]

print("Sold date flags:")
print(
    sold_clean[date_flag_columns]
    .sum()
    .to_string()
)

print("\nList date flags:")
print(
    list_clean[date_flag_columns]
    .sum()
    .to_string()
)

Sold date flags:
listing_after_close_flag      68
purchase_after_close_flag    241
negative_timeline_flag       290

List date flags:
listing_after_close_flag      84
purchase_after_close_flag    268
negative_timeline_flag       301


### 12. Duplicate-row review

In [176]:
print(
    "Sold fully duplicated rows:",
    sold_clean.duplicated().sum()
)

print(
    "List fully duplicated rows:",
    list_clean.duplicated().sum()
)

Sold fully duplicated rows: 48
List fully duplicated rows: 0


In [177]:
# create a function to check ListingKey
def review_listing_key_duplicates(df, dataset_name):
    if "ListingKey" not in df.columns:
        print(dataset_name, "does not contain ListingKey")
        return pd.DataFrame()

    duplicate_mask = df.duplicated(
        subset=["ListingKey"],
        keep=False
    )

    duplicate_records = (
        df.loc[duplicate_mask]
        .sort_values("ListingKey")
    )

    print(
        dataset_name,
        "rows with duplicated ListingKey:",
        len(duplicate_records)
    )

    print(
        dataset_name,
        "duplicated ListingKey count:",
        duplicate_records["ListingKey"].nunique()
    )

    return duplicate_records

In [178]:
sold_listing_key_duplicates = review_listing_key_duplicates(
    sold_clean,
    "Sold"
)

list_listing_key_duplicates = review_listing_key_duplicates(
    list_clean,
    "List"
)

Sold rows with duplicated ListingKey: 744
Sold duplicated ListingKey count: 370
List rows with duplicated ListingKey: 376
List duplicated ListingKey count: 187


We cannot remove by the same ListingKey because the data might come from different year and month dataset, but they have the same ListingKey.

In [204]:
duplicate_review_columns = [
    "ListingKey",
    "ListingId",
    "year_month",
    "source_file",
    "file_version",
    "MlsStatus",
    "ListPrice",
    "ClosePrice",
    "CloseDate"
]

existing_columns = [
    col for col in duplicate_review_columns
    if col in sold_listing_key_duplicates.columns
]

display(
    sold_listing_key_duplicates[existing_columns].head(10)
)

,ListingKey,ListingId,year_month,source_file,file_version,MlsStatus,ListPrice,ClosePrice,CloseDate
92419,1031992525,SW22259219,2024-06,CRMLSSold202406_filled.csv,filled,Closed,415740.0,415740.0,2024-06-21
157865,1031992525,SW22259219,2024-10,CRMLSSold202410.csv,original,Closed,459900.0,459900.0,2024-10-18
141580,1037464932,PW23082927,2024-09,CRMLSSold202409.csv,original,Closed,675000.0,665000.0,2024-09-06
75731,1037464932,PW23082927,2024-05,CRMLSSold202405_filled.csv,filled,Closed,675000.0,650000.0,2024-05-25
92367,1038376137,OC23108406,2024-06,CRMLSSold202406_filled.csv,filled,Closed,1732000.0,1695000.0,2024-06-10
75689,1038376137,OC23108406,2024-05,CRMLSSold202405_filled.csv,filled,Closed,1732000.0,1695000.0,2024-05-27
92081,1046689006,PI23203651,2024-06,CRMLSSold202406_filled.csv,filled,Closed,825000.0,825000.0,2024-06-30
157786,1046689006,PI23203651,2024-10,CRMLSSold202410.csv,original,Closed,825000.0,825000.0,2024-10-29
91945,1048638478,SR23217206,2024-06,CRMLSSold202406_filled.csv,filled,Closed,6700000.0,6600000.0,2024-06-25
126850,1048638478,SR23217206,2024-08,CRMLSSold202408.csv,original,Closed,6700000.0,6550000.0,2024-08-21


#### Remove fully duplicate rows

#### Sold

In [180]:
rows_before, cols_before = sold_clean.shape

sold_clean = sold_clean.drop_duplicates().copy()

rows_after, cols_after = sold_clean.shape

add_cleaning_log(
    dataset_name="Sold",
    step="Remove fully duplicated rows",
    rows_before=rows_before,
    rows_after=rows_after,
    cols_before=cols_before,
    cols_after=cols_after,
    notes="Removed rows identical across all columns"
)

#### List

In [181]:
rows_before, cols_before = list_clean.shape

list_clean = list_clean.drop_duplicates().copy()

rows_after, cols_after = list_clean.shape

add_cleaning_log(
    dataset_name="List",
    step="Remove fully duplicated rows",
    rows_before=rows_before,
    rows_after=rows_after,
    cols_before=cols_before,
    cols_after=cols_after,
    notes="Removed rows identical across all columns"
)

### 13. Remove invalid values

#### 13.1 Sold dataset removal standard

In [182]:
# Sold removal standard
sold_removal_flags = [
    "invalid_close_price_flag",
    "invalid_living_area_flag",
    "negative_days_on_market_flag",
    "negative_bedrooms_flag",
    "negative_bathrooms_flag"
]

sold_removal_flags = [
    col for col in sold_removal_flags
    if col in sold_clean.columns
]

sold_remove_mask = sold_clean[
    sold_removal_flags
].any(axis=1)

print("Sold rows proposed for removal:")
print(sold_remove_mask.sum())

Sold rows proposed for removal:
216


In [ ]:
# Check these flag's rows
sold_invalid_records = sold_clean.loc[
    sold_remove_mask
].copy()

display(sold_invalid_records.head(30))

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,...,out_of_state_flag,invalid_close_price_flag,invalid_list_price_flag,invalid_living_area_flag,negative_days_on_market_flag,negative_bedrooms_flag,negative_bathrooms_flag,listing_after_close_flag,purchase_after_close_flag,negative_timeline_flag
6,CoastalMendocino,CoastalMendocino,Tile,True,False,1950000.0,1063453216,kira@mendosir.com,2024-01-22,1950000.0,...,False,False,False,False,True,False,False,False,False,False
7708,Malibu,Malibu,Wood,True,<NA>,8295000.0,1046380894,benjamin@illulianrealty.com,2024-01-22,8000000.0,...,False,False,False,True,False,False,False,False,False,False
11214,NaN,NaN,"Carpet,Tile",True,True,899000.0,1062122119,thewalkers@bdhomes.com,2024-02-26,899000.0,...,False,False,False,False,True,False,False,False,False,False
11381,NaN,NaN,NaN,False,False,372000.0,1061302766,hello@yourdesertlife.org,2024-02-23,372000.0,...,False,False,False,False,True,False,False,False,False,False
18698,NaN,NaN,NaN,False,False,699900.0,1052574395,kristyglinski@gmail.com,2024-02-05,700000.0,...,False,False,False,False,True,False,False,False,False,False
21099,NaN,NaN,NaN,True,<NA>,9998000.0,1046973939,elizabeth@elizabethdonovanrealty.com,2024-02-09,9000000.0,...,False,False,False,True,False,False,False,False,False,False
22968,NaN,NaN,Wood,True,False,1950000.0,1045698780,tracy@tracydo.com,2024-02-23,1930000.0,...,False,False,False,True,False,False,False,False,False,False
22969,NaN,NaN,Wood,True,False,1995000.0,1045698270,tracy@tracydo.com,2024-02-23,1850000.0,...,False,False,False,True,False,False,False,False,False,False
23005,NaN,NaN,"Carpet,Vinyl",False,False,485410.0,1045653374,rhsaenz@drhorton.com,2024-02-29,476985.0,...,False,False,False,False,True,False,False,False,False,False
24397,NaN,NaN,Carpet,True,False,799000.0,1063549350,ainsleyhughes@kw.com,2024-03-21,799000.0,...,False,False,False,False,True,False,False,False,False,False


In [184]:
# remove Sold invalid values
rows_before, cols_before = sold_clean.shape

sold_clean = sold_clean.loc[
    ~sold_remove_mask
].copy()

rows_after, cols_after = sold_clean.shape

add_cleaning_log(
    dataset_name="Sold",
    step="Remove invalid numeric records",
    rows_before=rows_before,
    rows_after=rows_after,
    cols_before=cols_before,
    cols_after=cols_after,
    notes=", ".join(sold_removal_flags)
)

#### 13.2 List dataset removal standard

In [185]:
list_removal_flags = [
    "invalid_list_price_flag",
    "invalid_living_area_flag",
    "negative_days_on_market_flag",
    "negative_bedrooms_flag",
    "negative_bathrooms_flag"
]

list_removal_flags = [
    col for col in list_removal_flags
    if col in list_clean.columns
]

list_remove_mask = list_clean[
    list_removal_flags
].any(axis=1)

print("List rows proposed for removal:")
print(list_remove_mask.sum())

List rows proposed for removal:
423


In [186]:
# review removal rows
list_invalid_records = list_clean.loc[
    list_remove_mask
].copy()

display(list_invalid_records.head(30))

,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,long_zero_flag,out_of_state_flag,invalid_list_price_flag,invalid_living_area_flag,negative_days_on_market_flag,negative_bedrooms_flag,negative_bathrooms_flag,listing_after_close_flag,purchase_after_close_flag,negative_timeline_flag
162,799000.0,1063549350,ainsleyhughes@kw.com,2024-03-21,799000.0,Ainsley,Hughes,34.425577,-119.291855,11905 Silver Spur Street,...,False,False,False,False,True,False,False,False,False,False
166,899000.0,1063528331,absea@comcast.net,2024-03-19,810000.0,A.B.,Priceman DRE 0126...,39.383806,-123.789254,31530 Emerald Drive,...,False,False,False,False,True,False,False,False,False,False
873,1599000.0,1060153479,robert@anppros.com,2024-03-21,1625000.0,Robert,Perez,34.244449,-118.265167,3929 El Moreno Street,...,False,False,False,False,True,False,False,False,False,False
2725,469999.0,1059512539,khoren9@yahoo.com,NaT,NaN,Khoren,Barutyan,34.201700,-118.460053,15015 Sherman Way 103,...,False,False,False,False,True,False,False,False,False,False
2949,6495000.0,1059501249,TaniaFerris68@aol.com,NaT,NaN,Tania,Ferris,34.067738,-118.413460,605 Trenton Drive,...,False,False,False,True,False,False,False,False,False,False
6717,16000000.0,1058455780,brose@theagencyre.com,NaT,NaN,Billy,Rose,34.089788,-118.466626,1111 Linda Flora Drive,...,False,False,False,True,False,False,False,False,False,False
8819,498000.0,1058378873,Geary@MosaikRealEstate.com,NaT,NaN,Geary,Do,NaN,NaN,1930 Mission St 304,...,False,False,False,True,False,False,False,False,False,False
8852,17995000.0,1058377309,aperry@carolwoodre.com,NaT,NaN,Alexis,Perry,34.071606,-118.483070,735 N Bonhill Road,...,False,False,False,True,False,False,False,False,False,False
9274,12950000.0,1058311519,chris@chriscortazzo.com,NaT,NaN,Christopher,Cortazzo,34.081654,-118.936966,12815 Yellow Hill Road,...,False,False,False,True,False,False,False,False,False,False
9431,17750000.0,1058226711,info@davidkramer.com,NaT,NaN,David,Kramer,34.068878,-118.492920,1835 Old Orchard Road,...,False,False,False,True,False,False,False,False,False,False


In [187]:
# remove list invalid values
rows_before, cols_before = list_clean.shape

list_clean = list_clean.loc[
    ~list_remove_mask
].copy()

rows_after, cols_after = list_clean.shape

add_cleaning_log(
    dataset_name="List",
    step="Remove invalid numeric records",
    rows_before=rows_before,
    rows_after=rows_after,
    cols_before=cols_before,
    cols_after=cols_after,
    notes=", ".join(list_removal_flags)
)

### 14. Final validation

In [188]:
print("Final sold shape:", sold_clean.shape)
print("Final list shape:", list_clean.shape)

print("\nSold duplicate rows:")
print(sold_clean.duplicated().sum())

print("\nList duplicate rows:")
print(list_clean.duplicated().sum())

Final sold shape: (447769, 86)
Final list shape: (615316, 76)

Sold duplicate rows:
0

List duplicate rows:
0


#### 14.1 Check numeric dtype

In [189]:
existing_numeric_columns = [
    col for col in numeric_columns
    if col in sold_clean.columns
]

print(
    sold_clean[existing_numeric_columns]
    .dtypes
    .to_string()
)

ClosePrice               float64
ListPrice                float64
OriginalListPrice        float64
LivingArea               float64
LotSizeAcres             float64
BedroomsTotal            float64
BathroomsTotalInteger    float64
DaysOnMarket               int64
YearBuilt                float64
Latitude                 float64
Longitude                float64
ParkingTotal             float64


#### 14.2 Check date dtype

In [190]:
existing_date_columns = [
    col for col in date_columns
    if col in sold_clean.columns
]

print(
    sold_clean[existing_date_columns]
    .dtypes
    .to_string()
)

CloseDate                   datetime64[ns]
PurchaseContractDate        datetime64[ns]
ListingContractDate         datetime64[ns]
ContractStatusChangeDate    datetime64[ns]


#### 14.3 Create final missing value summaries

In [191]:
sold_missing_summary_after = create_missing_summary(
    sold_clean
)

list_missing_summary_after = create_missing_summary(
    list_clean
)

In [192]:
print("Cleaned Sold missing value summary")
print(sold_missing_summary_after)

print("Cleaned List missing value summary")
print(list_missing_summary_after)

Cleaned Sold missing value summary
                         column    dtype  non_null_count  missing_count  \
0       BuyerAgencyCompensation  float64           46095         401674   
1   BuyerAgencyCompensationType   object           46106         401663   
2              ElementarySchool   object           59512         388257   
3          MiddleOrJuniorSchool   object           59933         387836   
4                     lonfilled   object           63860         383909   
..                          ...      ...             ...            ...   
81               CountyOrParish   object          447769              0   
82                    MlsStatus   object          447769              0   
83                    ListingId   object          447769              0   
84                BedroomsTotal  float64          447757             12   
85       negative_timeline_flag     bool          447769              0   

    missing_percent  unique_count  
0             89.71         

#### 14.4 Check for Sold and List dataset columns

In [193]:
print("Sold cleaned columns:")

for col in sold_clean.columns:
    print(col)

print("\nList cleaned columns:")

for col in list_clean.columns:
    print(col)

Sold cleaned columns:
BuyerAgentAOR
ListAgentAOR
Flooring
ViewYN
PoolPrivateYN
OriginalListPrice
ListingKey
ListAgentEmail
CloseDate
ClosePrice
ListAgentFirstName
ListAgentLastName
Latitude
Longitude
UnparsedAddress
PropertyType
LivingArea
ListPrice
DaysOnMarket
ListOfficeName
BuyerOfficeName
CoListOfficeName
ListAgentFullName
CoListAgentFirstName
CoListAgentLastName
BuyerAgentMlsId
BuyerAgentFirstName
BuyerAgentLastName
AssociationFeeFrequency
ListingKeyNumeric
MLSAreaMajor
CountyOrParish
MlsStatus
ElementarySchool
AttachedGarageYN
ParkingTotal
PropertySubType
LotSizeAcres
SubdivisionName
BuyerOfficeAOR
YearBuilt
StreetNumberNumeric
ListingId
BathroomsTotalInteger
City
BedroomsTotal
ContractStatusChangeDate
PurchaseContractDate
ListingContractDate
StateOrProvince
MiddleOrJuniorSchool
FireplaceYN
Stories
HighSchool
Levels
LotSizeArea
MainLevelBedrooms
NewConstructionYN
GarageSpaces
HighSchoolDistrict
PostalCode
AssociationFee
LotSizeSquareFeet
OriginatingSystemName
OriginatingSystemSub

#### 14.5 Save Cleaning log

In [194]:
cleaning_log_df = pd.DataFrame(cleaning_log)

display(cleaning_log_df)

,dataset,step,rows_before,rows_after,rows_removed,columns_before,columns_after,columns_removed,notes
0,List,Remove duplicate .1 columns,615739,615739,0,89,78,11,"Dropped: ['PropertyType.1', 'ListAgentFirstNam..."
1,Sold,Remove fully duplicated rows,448033,447985,48,86,86,0,Removed rows identical across all columns
2,List,Remove fully duplicated rows,615739,615739,0,76,76,0,Removed rows identical across all columns
3,Sold,Remove invalid numeric records,447985,447769,216,86,86,0,"invalid_close_price_flag, invalid_living_area_..."
4,List,Remove invalid numeric records,615739,615316,423,76,76,0,"invalid_list_price_flag, invalid_living_area_f..."


In [195]:
# Export
cleaning_log_df.to_csv(
    CLEANED_DIR / "cleaning_log.csv",
    index=False
)

#### 14.6 Save Final Summary

In [197]:
sold_missing_summary_after.to_csv(
    CLEANED_DIR / "sold_missing_summary_after_cleaning.csv",
    index=False
)

list_missing_summary_after.to_csv(
    CLEANED_DIR / "list_missing_summary_after_cleaning.csv",
    index=False
)

### 15. Save Cleaned_Sold and Cleaned_List datasets

In [ ]:
sold_clean.to_csv(
    CLEANED_DIR / "combined_sold_residential_cleaned.csv",
    index=False
)

list_clean.to_csv(
    CLEANED_DIR / "combined_list_residential_cleaned.csv",
    index=False
)

In [198]:
# Make sure the data is successfully exported
print("Saved files:")

for file in CLEANED_DIR.iterdir():
    print(file.name)

Saved files:
list_missing_summary_after_cleaning.csv
combined_list_residential_cleaned.csv
combined_sold_residential_cleaned.csv
cleaning_log.csv
sold_missing_summary_after_cleaning.csv
